In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/notebooks/arijitmohanta/02-preprocessing/__results__.html
/kaggle/input/notebooks/arijitmohanta/02-preprocessing/X_feat.pkl
/kaggle/input/notebooks/arijitmohanta/02-preprocessing/__notebook__.ipynb
/kaggle/input/notebooks/arijitmohanta/02-preprocessing/y.npy
/kaggle/input/notebooks/arijitmohanta/02-preprocessing/X_img.npy
/kaggle/input/notebooks/arijitmohanta/02-preprocessing/__output__.json
/kaggle/input/notebooks/arijitmohanta/02-preprocessing/custom.css
/kaggle/input/notebooks/arijitmohanta/02-preprocessing/__results___files/__results___4_0.png
/kaggle/input/notebooks/arijitmohanta/02-preprocessing/__results___files/__results___2_0.png


In [2]:
# ============================================================
# STAGE 3.1 — LOCATE THE STAGE 2 ARTIFACTS ON KAGGLE
# The Stage 2 output mounts read-only somewhere under /kaggle/input,
# but the exact folder ("slug") depends on the Stage 2 notebook title.
#       1. Walk the whole input tree and print every file with its size,
#       2. Read the sizes as a sanity check, and
#       3. Pull out the full paths of the three artifacts. Step 2 loads.
# ============================================================

import os                                     # stdlib; the only tool needed to walk the mount

# Walk every folder under /kaggle/input and print each file with its size in MB.
# The size column is a free integrity check: the image tensor is the large file
# and the other two are small, so a wildly wrong size means the wrong output got
# attached, or Stage 2 didn't save cleanly.
print("Full listing under /kaggle/input:\n")
for dirpath, dirnames, filenames in os.walk('/kaggle/input'):      # recurse the whole mount
    for f in filenames:
        full = os.path.join(dirpath, f)                            # rebuild the absolute path
        print(f"{os.path.getsize(full) / 1e6:8.1f} MB   {full}")   # size (MB) + path, aligned

# Scan the tree a second time, keeping only the three files we actually need, so
# Step 2 can reference these resolved paths directly instead of you retyping a
# long /kaggle/input/... string by hand.
targets = ['X_img.npy', 'X_feat.pkl', 'y.npy']     # the row-aligned Stage 2 outputs
found = {}                                          # filename -> resolved absolute path
for dirpath, dirnames, filenames in os.walk('/kaggle/input'):
    for f in filenames:
        if f in targets:                            # exact filename match, ignore everything else
            found[f] = os.path.join(dirpath, f)

print("\nTargets found:")
for t in targets:
    print(f"  {t}: {found.get(t, 'NOT FOUND')}")    # 'NOT FOUND' flags an unattached input

Full listing under /kaggle/input:

     0.4 MB   /kaggle/input/notebooks/arijitmohanta/02-preprocessing/__results__.html
    22.1 MB   /kaggle/input/notebooks/arijitmohanta/02-preprocessing/X_feat.pkl
     0.2 MB   /kaggle/input/notebooks/arijitmohanta/02-preprocessing/__notebook__.ipynb
     6.2 MB   /kaggle/input/notebooks/arijitmohanta/02-preprocessing/y.npy
   708.4 MB   /kaggle/input/notebooks/arijitmohanta/02-preprocessing/X_img.npy
     0.0 MB   /kaggle/input/notebooks/arijitmohanta/02-preprocessing/__output__.json
     0.0 MB   /kaggle/input/notebooks/arijitmohanta/02-preprocessing/custom.css
     0.1 MB   /kaggle/input/notebooks/arijitmohanta/02-preprocessing/__results___files/__results___4_0.png
     0.1 MB   /kaggle/input/notebooks/arijitmohanta/02-preprocessing/__results___files/__results___2_0.png

Targets found:
  X_img.npy: /kaggle/input/notebooks/arijitmohanta/02-preprocessing/X_img.npy
  X_feat.pkl: /kaggle/input/notebooks/arijitmohanta/02-preprocessing/X_feat.pkl
  y.

In [3]:
# ============================================================
# STAGE 3.2 — LOAD THE STAGE 2 ARTIFACTS + RE-ASSERT ALIGNMENT
# Bring the three row-aligned objects back into memory:
#       1. X_img  — the (N, 64, 64) uint8 image tensor  (CNN route),
#       2. X_feat — the (N, 16) named feature table      (classical route),
#       3. y      — the (N,) string labels               (shared by both).
# Re-run the alignment check across all three
# A save/reload boundary is exactly where row order can silently drift.
# ============================================================

import numpy as np                            # arrays + .npy loading
import pandas as pd                            # the feature table is a DataFrame

# One base path, resolved from Step 1, so the three loads read from the same confirmed folder
# There's a single place to change if the input is ever re-attached under a different slug.
BASE = '/kaggle/input/notebooks/arijitmohanta/02-preprocessing'   # confirmed in Step 1

# Load each artifact in its native format: 
#          -- .npy for the two arrays (fast binary, preserves dtype)
#          -- pickle for the DataFrame (preserves the 16 column names, which np.save would have thrown away).
X_img  = np.load(f'{BASE}/X_img.npy')          # (172950, 64, 64) uint8 image cube
y      = np.load(f'{BASE}/y.npy', allow_pickle=True)   # object array of string labels
X_feat = pd.read_pickle(f'{BASE}/X_feat.pkl')  # (172950, 16) named feature table

# Report shape + dtype of each so we can eyeball that nothing loaded, truncated, or in the wrong type (e.g., X_img must stay uint8, not float).
print("X_img :", X_img.shape,  X_img.dtype)    # expect (172950, 64, 64) uint8
print("y     :", y.shape,      y.dtype)        # expect (172950,) object
print("X_feat:", X_feat.shape)                 # expect (172950, 16)

# The 16 feature names
# So we confirm the column order survived the pickle and can plan the Stage 3 drops (radon_angle_strength duplicate, dead radial_ring_9/10).
print("\nFeature columns:")
print(list(X_feat.columns))

# ALIGNMENT CHECK — the non-negotiable one, repeated post-reload.
# If the three row counts ever disagree, let's stop hard here 
# rather than let mismatched rows reach the split and poison every per-class metric downstream.
assert X_img.shape[0] == len(y) == len(X_feat), "Row-count mismatch across X_img / y / X_feat!"
print("\nAlignment OK — all three have", X_img.shape[0], "rows.")

X_img : (172950, 64, 64) uint8
y     : (172950,) <U9
X_feat: (172950, 16)

Feature columns:
['radial_ring_1', 'radial_ring_2', 'radial_ring_3', 'radial_ring_4', 'radial_ring_5', 'radial_ring_6', 'radial_ring_7', 'radial_ring_8', 'radial_ring_9', 'radial_ring_10', 'radon_peak', 'radon_peak_ratio', 'radon_angle_strength', 'conn_largest_frac', 'conn_fragmentation', 'angular_peak_bin']

Alignment OK — all three have 172950 rows.


In [4]:
# ============================================================
# STAGE 3.3 — PRUNE DEAD / DUPLICATE FEATURES (classical route)
# Three of the 16 hand features carry no usable signal. Aim:
#       1. PROVE each one is dead before removing it (verify, don't assume),
#       2. Drop the three columns, and
#       3. Confirm the pruned table is (N, 13) with the right names.
# Only X_feat is affected — the CNN route (X_img) uses raw pixels.
# ============================================================

# --- 1. Prove radon_angle_strength is an exact duplicate of radon_peak ---
# Stage 2 computed peak as sino.max() and angle_strength as sino.max(axis=0).max();
# the column-max-then-max equals the global max, so they must be identical on every row.
# We assert exact equality so that if this is ever NOT true, the drop halts instead of silently discarding a column that turned out to differ.
dup_identical = (X_feat['radon_angle_strength'] == X_feat['radon_peak']).all()
print("radon_angle_strength == radon_peak on every row:", dup_identical)
assert dup_identical, "radon_angle_strength is NOT a perfect duplicate — investigate before dropping!"

# --- 2. Show radial_ring_9 / _10 are structurally empty, and ring_8 is not ---
# The radial profile normalizes distance to the SQUARE frame's corner
# The outer rings 9-10 fall in the frame corners that lie outside the circular wafer (no dies -> failure rate 0 for every wafer).
# Ring 8 straddles the physical wafer edge, so it still carries the edge signal.
# Printing all three side by side makes the "dead vs live" boundary visible rather than asserted.
for col in ['radial_ring_8', 'radial_ring_9', 'radial_ring_10']:
    print(f"{col:16s} max={X_feat[col].max():.4f}   nonzero rows={int((X_feat[col] != 0).sum()):,}")

# --- 3. Drop the three dead columns ---
# radon_angle_strength (perfect duplicate) + the two empty outer rings.
# Dropping the duplicate also protects Stage 4's feature-importance readout:
# two identical columns would split one feature's importance in half and understate radon_peak.
drop_cols = ['radon_angle_strength', 'radial_ring_9', 'radial_ring_10']
X_feat = X_feat.drop(columns=drop_cols)        # reassign to the pruned 13-column table

print("\nPruned feature table:", X_feat.shape)  # expect (172950, 13)
print("Remaining columns:")
print(list(X_feat.columns))

radon_angle_strength == radon_peak on every row: True
radial_ring_8    max=1.0000   nonzero rows=172,623
radial_ring_9    max=1.0000   nonzero rows=4,164
radial_ring_10   max=1.0000   nonzero rows=888

Pruned feature table: (172950, 13)
Remaining columns:
['radial_ring_1', 'radial_ring_2', 'radial_ring_3', 'radial_ring_4', 'radial_ring_5', 'radial_ring_6', 'radial_ring_7', 'radial_ring_8', 'radon_peak', 'radon_peak_ratio', 'conn_largest_frac', 'conn_fragmentation', 'angular_peak_bin']


In [6]:
# ============================================================
# STAGE 3.3 — FEATURE PRUNING (with a corrected assumption on record)
#
# Goal: remove hand features that carry no signal before modeling.
# Three columns were candidates. Only one was actually dead.
#
# WHAT HAPPENED (kept on record deliberately):
#   The first version of this step dropped THREE columns:
#   radon_angle_strength, radial_ring_9, and radial_ring_10.
#   The duplicate drop was correct. The two-ring drops were NOT.
#
#   The assumption behind dropping rings 9/10 was geometric:
#   the radial profile bins distance out to the corner of the square 64x64 frame
#   for a circular wafer inscribed in that square —
#   the outermost rings should fall in empty frame corners (no dies, failure rate 0 on every wafer). 
#   Plausible, and it was carried forward from the Stage 2 notes as "always zero".
#
#   The verification step FALSIFIED it. 
#   Rings 9/10 are nonzero for 4,164 and 888 wafers, respectively, with rates up to 1.0.
#   Cause:
#   the profile normalizes distance to the farthest die in EACH map,
#   not to a fixed disc, and WM-811K contains 346 distinct map shapes.
#   Non-square, off-center, and partial maps place real dies at the
#   bounding-box corner (normalized radius > 0.8), so those rings fire.
#   A ring 9/10 signal therefore encodes "this map's dies reach its
#   corner" — a real, if minority, structural cue, not void.
#
#   DECISION: keep rings 9/10 and let Stage 4 feature importance judge
#   them on evidence; drop only the proven duplicate. The lesson worth
#   showing: every drop is verified in code first, precisely because a
#   reasonable-sounding assumption was wrong across ~5,000 wafers.
#
# This cell reloads the untouched 16-column table and drops exactly one
# column, yielding a clean (N, 15) feature set.
# ============================================================

# Reload the original feature table so we start from all 16 columns again, rather
# than trying to un-drop from the mutated 13-column object now in memory.
X_feat = pd.read_pickle(f'{BASE}/X_feat.pkl')   # BASE resolved in Step 2

# --- Re-prove the duplicate before removing it ---
# radon_peak = sino.max(), radon_angle_strength = sino.max(axis=0).max(); these
# are arithmetically identical, so we assert exact equality and halt if it ever
# isn't true, rather than silently discarding a column that turned out to differ.
dup_identical = (X_feat['radon_angle_strength'] == X_feat['radon_peak']).all()
assert dup_identical, "radon_angle_strength is NOT a perfect duplicate — investigate before dropping!"

# --- Leave the retention evidence in the notebook output ---
# Printing the nonzero counts is what turns "we kept rings 9/10" from an opinion into a record: 
# A reader sees the exact wafer counts that justified keeping them.
for col in ['radial_ring_9', 'radial_ring_10']:
    print(f"KEEP {col:15s} nonzero rows={int((X_feat[col] != 0).sum()):,}  (carries signal)")

# --- Drop only the proven duplicate ---
# Removing it also protects Stage 4's feature-importance readout: 
# two identical columns would split radon_peak's importance in half and understate it.
X_feat = X_feat.drop(columns=['radon_angle_strength'])   # 16 -> 15 columns

print("\nPruned feature table:", X_feat.shape)            # expect (172950, 15)
print("Remaining columns:")
print(list(X_feat.columns))

KEEP radial_ring_9   nonzero rows=4,164  (carries signal)
KEEP radial_ring_10  nonzero rows=888  (carries signal)

Pruned feature table: (172950, 15)
Remaining columns:
['radial_ring_1', 'radial_ring_2', 'radial_ring_3', 'radial_ring_4', 'radial_ring_5', 'radial_ring_6', 'radial_ring_7', 'radial_ring_8', 'radial_ring_9', 'radial_ring_10', 'radon_peak', 'radon_peak_ratio', 'conn_largest_frac', 'conn_fragmentation', 'angular_peak_bin']


In [7]:
# ============================================================
# STAGE 3.4 — STRATIFIED TRAIN / TEST SPLIT (shared by both routes)
# Replaces the dataset's official split (which was inverted — Test
# larger than Train). We
#       1. Split ROW INDICES with stratification on y, so every class
#          holds its proportion in both partitions,
#       2. Apply the same indices to X_img, X_feat, and y, so the CNN
#          and classical routes train/test on the identical wafers,
#       3. Verify each class is present on both sides with a matched %.
# 80/20 split, fixed seed for reproducibility.
# ============================================================

from sklearn.model_selection import train_test_split   # stratify= gives class-proportional splits

# Split an ARRAY OF INDICES (0..N-1) rather than the data itself. 
# Splitting indices once and reusing them keeps X_img / X_feat / y perfectly aligned across both routes
# The alternative (splitting each array separately) risks the two routes landing on different partitions and quietly leaking test wafers.
idx = np.arange(len(y))                                 # one index per wafer, in row order

# stratify=y is the whole point: it forces each class to appear in train and test at its full-set proportion, 
# so Near-full (149) and Donut (555) can't vanish from the test set and leave macro-F1 undefined on those classes.
train_idx, test_idx = train_test_split(
    idx,                                                # split the indices, not the arrays
    test_size=0.20,                                     # 80% train / 20% test
    stratify=y,                                         # class-proportional on every label
    random_state=42                                     # fixed seed -> identical split every run
)

# Apply the SAME indices to all three artifacts. Fancy-indexing a numpy array and
# .iloc on the DataFrame both preserve order, so row i stays the same wafer across
# X_img, X_feat, and y on each side of the split.
X_img_train,  X_img_test  = X_img[train_idx],        X_img[test_idx]
X_feat_train, X_feat_test = X_feat.iloc[train_idx],  X_feat.iloc[test_idx]
y_train,      y_test      = y[train_idx],            y[test_idx]

print("Train:", len(train_idx), " Test:", len(test_idx))

# --- Verify stratification held ---
# Build a side-by-side table of each class's share (%) in train vs test.
# split worked --> the two percentage columns match to ~2 decimals for every class,
# including the rare ones — that's the proof the rare classes survived intact.
import pandas as pd
train_pct = pd.Series(y_train).value_counts(normalize=True).mul(100).round(2)
test_pct  = pd.Series(y_test ).value_counts(normalize=True).mul(100).round(2)
check = pd.DataFrame({'train_%': train_pct, 'test_%': test_pct})
check['test_n'] = pd.Series(y_test).value_counts()      # raw test count — must be >0 for every class
print("\nClass share, train vs test (should match per row):")
print(check.sort_values('train_%', ascending=False))

Train: 138360  Test: 34590

Class share, train vs test (should match per row):
           train_%  test_%  test_n
none         85.25   85.24   29486
Edge-Ring     5.60    5.60    1936
Edge-Loc      3.00    3.00    1038
Center        2.48    2.48     859
Loc           2.08    2.08     718
Scratch       0.69    0.69     239
Random        0.50    0.50     173
Donut         0.32    0.32     111
Near-full     0.09    0.09      30


In [8]:
# ============================================================
# STAGE 3.5 — BALANCED CLASS WEIGHTS (classical route imbalance handling)
# Instead of resampling, we penalize mistakes on rare classes more heavily at training time. 
# This cell
#       1. computes sklearn's 'balanced' weight for each class,
#       2. builds a {label: weight} dict to pass to models in Stage 4,
#       3. Inspects the multipliers and sanity-checks their spread.
# Weights are derived from TRAIN ONLY — test keeps the real ~989:1 distribution, since that's the wafer mix the fab actually runs.
# ============================================================

from sklearn.utils.class_weight import compute_class_weight   # the 'balanced' heuristic, made explicit

# The 'balanced' rule sets each class's weight to n_samples / (n_classes * n_class),
# so a class's weight rises as it gets rarer. 
# We compute it on y_train only and key it by the sorted class names, so the dict maps each label to its penalty.
classes = np.unique(y_train)                                  # 9 class names, sorted
weights = compute_class_weight(                              # one weight per class, same order as `classes`
    class_weight='balanced',
    classes=classes,
    y=y_train                                                # TRAIN labels only — never test
)
class_weights = dict(zip(classes, weights))                  # {label: weight}, ready for Stage 4 .fit()

# Lay the weights next to each class's train count so the multipliers are legible
# The common class is the only one pushed below 1.0 (down-weighted).
train_counts = pd.Series(y_train).value_counts()
table = pd.DataFrame({
    'train_n': train_counts,
    'weight':  pd.Series(class_weights)
}).sort_values('weight')                                     # ascending: none first, Near-full last
print("Balanced class weights (train-derived):")
print(table.round(3))

# Sanity check: the ratio of the largest to smallest weight should recover the raw imbalance ratio (~989:1). 
# If it does, the weighting is scaling exactly with rarity — the confirmation that the weights encode the imbalance faithfully.
ratio = table['weight'].max() / table['weight'].min()
print(f"\nWeight spread (max/min): {ratio:.0f}x   — should track the ~989:1 imbalance")

Balanced class weights (train-derived):
           train_n   weight
none        117945    0.130
Edge-Ring     7744    1.985
Edge-Loc      4151    3.704
Center        3435    4.475
Loc           2875    5.347
Scratch        954   16.115
Random         693   22.184
Donut          444   34.625
Near-full      119  129.188

Weight spread (max/min): 991x   — should track the ~989:1 imbalance


In [9]:
# ============================================================
# STAGE 3.6 — SAVE STAGE 3 ARTIFACTS FOR STAGE 4
# Persist everything Stage 4 needs to reproduce this exact setup:
#       1. X_feat_pruned.pkl — the (N, 15) feature table (duplicate dropped),
#       2. train_idx.npy / test_idx.npy — the stratified split indices,
#       3. class_weights.pkl — the {label: weight} dict.
# Saving INDICES (not pre-split arrays) keeps both routes provably on the same wafers.
# X_img and y are NOT re-saved — Stage 4 re-attaches Stage 2's copies and applies these indices to them.
# ============================================================

import pickle                                    # for the weight dict; np.save can't store a dict cleanly

# The pruned feature table goes to pickle to preserve its 15 column names, which
# are what make Stage 4's feature-importance readout human-readable.
X_feat.to_pickle('/kaggle/working/X_feat_pruned.pkl')       # (172950, 15) named table

# The split indices are plain integer arrays — .npy is the fast, exact format.
# Stage 4 loads these and does X_img[train_idx] etc. to rebuild the identical split.
np.save('/kaggle/working/train_idx.npy', train_idx)          # 138360 train row positions
np.save('/kaggle/working/test_idx.npy',  test_idx)           # 34590 test row positions

# The weight dict is a Python object, so it goes to pickle. Stage 4 passes it
# straight into class_weight= for the RF/SVM and into the CNN's loss.
with open('/kaggle/working/class_weights.pkl', 'wb') as f:
    pickle.dump(class_weights, f)

# Read every file back and print its size, confirming all five actually wrote
import os
for f in ['X_feat_pruned.pkl', 'train_idx.npy', 'test_idx.npy', 'class_weights.pkl']:
    kb = os.path.getsize(f'/kaggle/working/{f}') / 1024
    print(f"{f:22s} {kb:9.1f} KB")

print("\nSaved. Stage 3 complete — ready for Stage 4 modelling.")

X_feat_pruned.pkl        20268.5 KB
train_idx.npy             1081.1 KB
test_idx.npy               270.4 KB
class_weights.pkl            0.8 KB

Saved. Stage 3 complete — ready for Stage 4 modelling.
